In [ ]:
# Cell 1 — locate and validate the saved Day 7 white-dwarf simulations.
from pathlib import Path
import json, time, math, copy, random
import numpy as np
import pandas as pd
import torch
from torch import nn

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/day15-white-dwarf-flow')
OUT.mkdir(parents=True, exist_ok=True)

matches = []
for meta_path in INPUT.rglob('simulation_metadata.json'):
    try:
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        root = meta_path.parent
        if (meta.get('family') == 'white_dwarf'
                and (root / 'observations.npy').is_file()
                and (root / 'parameters.csv').is_file()):
            matches.append((int(meta.get('parameter_draws_unique', 0)), root, meta))
    except Exception as exc:
        print('Skipping unreadable metadata:', meta_path, type(exc).__name__)

if not matches:
    raise FileNotFoundError(
        'No saved white-dwarf simulation found. Add the successful Day 7 Physics '
        'notebook output as an input; it must include simulations/white_dwarf/ '
        'with observations.npy, parameters.csv, and simulation_metadata.json.'
    )
matches.sort(key=lambda item: (item[0], str(item[1])))
N_DRAWS, SIM_DIR, META = matches[-1]
print('[PASS] Selected saved Day 7 white-dwarf simulations:', SIM_DIR)
print('[PASS] Unique parameter draws:', N_DRAWS,
      '| simulation records:', META.get('simulated_observation_count'),
      '| input:', META.get('input_representation'))

if N_DRAWS < 100_000:
    print('[WARNING] This input has fewer than 100,000 unique white-dwarf draws.')

PARAMS = pd.read_csv(SIM_DIR / 'parameters.csv')
OBS = np.load(SIM_DIR / 'observations.npy', mmap_mode='r', allow_pickle=False)
required = {'parameter_id', 'mass_solar', 'radius_error_solar'}
missing = sorted(required.difference(PARAMS.columns))
if missing:
    raise ValueError(f'White-dwarf parameter table is missing columns: {missing}')
if OBS.ndim != 2 or OBS.shape[1] != 2 or len(PARAMS) != len(OBS):
    raise ValueError(f'Expected aligned observations with shape (N, 2); got {OBS.shape} and {len(PARAMS)} rows')
if not np.isfinite(PARAMS['mass_solar'].to_numpy()).all() or not np.isfinite(OBS).all():
    raise ValueError('Input simulations contain non-finite values')
if (OBS[:, 1] <= 0).any():
    raise ValueError('Every radius uncertainty must be positive')

mass = PARAMS['mass_solar'].to_numpy(dtype=np.float32)
groups = PARAMS['parameter_id'].to_numpy()
observed = np.asarray(OBS, dtype=np.float32)
print(f'[PASS] Aligned records={len(mass):,}; unique parameter IDs={np.unique(groups).size:,}; '
      f'observation columns={META.get("observation_columns")}')


In [ ]:
# Cell 2 — fit the conditional flow and evaluate on untouched parameter groups.
from IPython.display import display
import matplotlib.pyplot as plt

SEED = 1515
MASS_LOW, MASS_HIGH = 0.15, 1.35  # same uniform mass prior as the Day 7 simulator / NUTS model
EPOCHS = 120
BATCH_SIZE = 4096
PATIENCE = 12
LEARNING_RATE = 1e-3
POSTERIOR_Z90 = 1.6448536269514722
POSTERIOR_Z50 = 0.6744897501960817

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[INFO] Torch device:', DEVICE)

# Split by parameter_id, not by row, to prevent noise replicas of one truth leaking.
unique_groups = np.unique(groups)
rng = np.random.default_rng(SEED)
shuffled = rng.permutation(unique_groups)
n_train = int(0.80 * len(shuffled))
n_valid = int(0.10 * len(shuffled))
train_groups = shuffled[:n_train]
valid_groups = shuffled[n_train:n_train+n_valid]
test_groups = shuffled[n_train+n_valid:]
train_idx = np.flatnonzero(np.isin(groups, train_groups))
valid_idx = np.flatnonzero(np.isin(groups, valid_groups))
test_idx = np.flatnonzero(np.isin(groups, test_groups))
print(f'[PASS] Group-disjoint split: train={len(train_groups):,} groups/{len(train_idx):,} rows; '
      f'validation={len(valid_groups):,}/{len(valid_idx):,}; test={len(test_groups):,}/{len(test_idx):,}')

# A logit maps the bounded mass prior (0.15, 1.35) to the real line.
EPS = 1e-6
unit_mass = np.clip((mass - MASS_LOW) / (MASS_HIGH - MASS_LOW), EPS, 1.0-EPS)
y = np.log(unit_mass) - np.log1p(-unit_mass)
y_mean = float(y[train_idx].mean())
y_std = float(y[train_idx].std())
if not np.isfinite(y_std) or y_std < 1e-6:
    raise ValueError('Training target has invalid scale')
y_scaled = ((y - y_mean) / y_std).astype(np.float32)

# Context is the observed radius and its reported 1-sigma uncertainty.
x_mean = observed[train_idx].mean(axis=0, keepdims=True)
x_std = observed[train_idx].std(axis=0, keepdims=True)
x_std = np.where(x_std < 1e-8, 1.0, x_std)
x_scaled = ((observed - x_mean) / x_std).astype(np.float32)

class ConditionalAffineFlow(nn.Module):
    # 1-D conditional flow: affine Gaussian latent followed by bounded logistic mass transform.
    def __init__(self, context_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(context_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2),
        )
    def forward(self, context):
        mu, log_sigma = self.net(context).chunk(2, dim=-1)
        log_sigma = torch.clamp(log_sigma, min=-4.0, max=2.0)
        return mu.squeeze(-1), log_sigma.squeeze(-1)
    def nll(self, target, context):
        mu, log_sigma = self(context)
        inv_var = torch.exp(-2.0 * log_sigma)
        return (0.5 * (target-mu).square() * inv_var + log_sigma + 0.5*math.log(2.0*math.pi)).mean()

model = ConditionalAffineFlow(context_dim=2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
tx = torch.from_numpy(x_scaled)
ty = torch.from_numpy(y_scaled)
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(tx[train_idx], ty[train_idx]),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
)
valid_x = tx[valid_idx].to(DEVICE)
valid_y = ty[valid_idx].to(DEVICE)
history = []
best_state, best_valid, best_epoch, stale = None, float('inf'), 0, 0
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        loss = model.nll(yb, xb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        batch_losses.append(float(loss.detach().cpu()))
    model.eval()
    with torch.no_grad():
        valid_nll = float(model.nll(valid_y, valid_x).cpu())
    train_nll = float(np.mean(batch_losses))
    history.append({'epoch': epoch+1, 'train_nll': train_nll, 'validation_nll': valid_nll})
    if valid_nll < best_valid - 1e-5:
        best_valid, best_epoch, stale = valid_nll, epoch+1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stale += 1
    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:03d}: train NLL={train_nll:.4f}; validation NLL={valid_nll:.4f}; best={best_epoch}')
    if stale >= PATIENCE:
        print(f'[INFO] Early stopping after {epoch+1} epochs; best epoch={best_epoch}')
        break
if best_state is None:
    raise RuntimeError('Training did not produce a best validation checkpoint')
model.load_state_dict(best_state)
model.eval()
print(f'[PASS] Restored best validation checkpoint at epoch {best_epoch}; '
      f'elapsed={time.time()-start_time:.1f}s')

# Analytic posterior intervals from the conditional flow's Normal latent distribution.
with torch.no_grad():
    test_x_t = tx[test_idx].to(DEVICE)
    mu_t, log_sigma_t = model(test_x_t)
    mu = mu_t.cpu().numpy()
    sigma = np.exp(log_sigma_t.cpu().numpy())
def latent_to_mass(z_scaled):
    z = y_mean + y_std * z_scaled
    # Stable logistic transform.
    u = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
    return MASS_LOW + (MASS_HIGH-MASS_LOW) * u
flow_q05 = latent_to_mass(mu - POSTERIOR_Z90*sigma)
flow_q95 = latent_to_mass(mu + POSTERIOR_Z90*sigma)
flow_q25 = latent_to_mass(mu - POSTERIOR_Z50*sigma)
flow_q75 = latent_to_mass(mu + POSTERIOR_Z50*sigma)
flow_median = latent_to_mass(mu)
truth = mass[test_idx]
test_obs = observed[test_idx]
test_group_ids = groups[test_idx]

# Numerical posterior reference: p(m|r,s) ∝ Uniform(m) × Normal(r; R_Nauenberg(m), s).
def nauenberg_radius(m):
    mch = 1.44
    return 0.0112 * np.sqrt(np.maximum((mch/m)**(2.0/3.0) - (m/mch)**(2.0/3.0), 0.0))

grid = np.linspace(MASS_LOW, MASS_HIGH, 4097, dtype=np.float64)
grid_radius = nauenberg_radius(grid)
ref_q05 = np.empty(len(test_idx)); ref_median = np.empty(len(test_idx)); ref_q95 = np.empty(len(test_idx))
for lo in range(0, len(test_idx), 256):
    hi = min(lo+256, len(test_idx))
    r_obs = test_obs[lo:hi, 0].astype(np.float64)[:, None]
    r_err = test_obs[lo:hi, 1].astype(np.float64)[:, None]
    logp = -0.5*((r_obs-grid_radius[None, :])/r_err)**2 - np.log(r_err)
    logp -= np.max(logp, axis=1, keepdims=True)
    density = np.exp(logp)
    cdf = np.cumsum(density, axis=1)
    cdf /= cdf[:, -1:]
    for q, dest in ((0.05, ref_q05), (0.50, ref_median), (0.95, ref_q95)):
        dest[lo:hi] = np.asarray([np.interp(q, row, grid) for row in cdf])

def group_bootstrap_ci(values, group_ids, seed=73, reps=1000):
    frame = pd.DataFrame({'group': group_ids, 'value': values}).groupby('group', sort=False)['value'].mean().to_numpy()
    rng_ci = np.random.default_rng(seed)
    estimates = np.empty(reps)
    for j in range(reps):
        estimates[j] = rng_ci.choice(frame, size=len(frame), replace=True).mean()
    return float(np.quantile(estimates, 0.025)), float(np.quantile(estimates, 0.975))

cover50 = (truth >= flow_q25) & (truth <= flow_q75)
cover90 = (truth >= flow_q05) & (truth <= flow_q95)
ref_cover90 = (truth >= ref_q05) & (truth <= ref_q95)
ci50 = group_bootstrap_ci(cover50.astype(float), test_group_ids, seed=51)
ci90 = group_bootstrap_ci(cover90.astype(float), test_group_ids, seed=52)
ci_ref90 = group_bootstrap_ci(ref_cover90.astype(float), test_group_ids, seed=53)

coverage = pd.DataFrame([
    {'method':'conditional_affine_flow', 'nominal_coverage':0.50, 'empirical_coverage':float(cover50.mean()),
     'group_bootstrap_95_low':ci50[0], 'group_bootstrap_95_high':ci50[1],
     'median_interval_width_solar':float(np.median(flow_q75-flow_q25))},
    {'method':'conditional_affine_flow', 'nominal_coverage':0.90, 'empirical_coverage':float(cover90.mean()),
     'group_bootstrap_95_low':ci90[0], 'group_bootstrap_95_high':ci90[1],
     'median_interval_width_solar':float(np.median(flow_q95-flow_q05))},
    {'method':'numerical_grid_reference', 'nominal_coverage':0.90, 'empirical_coverage':float(ref_cover90.mean()),
     'group_bootstrap_95_low':ci_ref90[0], 'group_bootstrap_95_high':ci_ref90[1],
     'median_interval_width_solar':float(np.median(ref_q95-ref_q05))},
])

comparison = pd.DataFrame({
    'parameter_id':test_group_ids,
    'mass_truth_solar':truth,
    'observed_radius_solar':test_obs[:,0],
    'radius_uncertainty_solar':test_obs[:,1],
    'flow_q05_solar':flow_q05, 'flow_median_solar':flow_median, 'flow_q95_solar':flow_q95,
    'grid_q05_solar':ref_q05, 'grid_median_solar':ref_median, 'grid_q95_solar':ref_q95,
    'flow_truth_in_90pct_interval':cover90,
    'grid_truth_in_90pct_interval':ref_cover90,
})

torch.save({
    'state_dict': model.cpu().state_dict(),
    'model': 'conditional affine normalizing flow, one-dimensional logit-mass latent',
    'family': 'white_dwarf',
    'mass_prior_bounds_solar': [MASS_LOW, MASS_HIGH],
    'target_logit_mean': y_mean, 'target_logit_std': y_std,
    'context_mean': x_mean.astype(np.float32), 'context_std': x_std.astype(np.float32),
    'context_columns': ['observed_radius_solar','radius_uncertainty_solar'],
    'seed': SEED,
}, OUT/'day15_white_dwarf_conditional_flow.pt')
pd.DataFrame(history).to_csv(OUT/'day15_white_dwarf_flow_training_history.csv', index=False)
coverage.to_csv(OUT/'day15_white_dwarf_flow_coverage.csv', index=False)
comparison.to_csv(OUT/'day15_white_dwarf_flow_posterior_comparison.csv', index=False)

flow_median_abs_error = float(np.median(np.abs(flow_median-ref_median)))
median_width_ratio = float(np.median(flow_q95-flow_q05) / np.median(ref_q95-ref_q05))
summary = {
    'created_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'source_simulation_dir': str(SIM_DIR),
    'source_metadata': META,
    'family': 'white_dwarf',
    'model': 'one-dimensional conditional affine normalizing flow; Gaussian base transformed through a learned affine map and bounded logistic mass transform',
    'target': 'mass_solar',
    'context': ['observed_radius_solar','radius_uncertainty_solar'],
    'prior_bounds_solar': [MASS_LOW, MASS_HIGH],
    'split': {'unit':'parameter_id groups','train_groups':int(len(train_groups)), 'validation_groups':int(len(valid_groups)), 'untouched_test_groups':int(len(test_groups)),
              'train_rows':int(len(train_idx)), 'validation_rows':int(len(valid_idx)), 'test_rows':int(len(test_idx))},
    'training': {'device':str(DEVICE),'epochs_requested':EPOCHS,'epochs_run':len(history),'best_epoch':int(best_epoch),
                 'best_validation_nll':float(best_valid),'seconds':float(time.time()-start_time)},
    'flow_coverage': {'50pct':float(cover50.mean()),'50pct_group_bootstrap_95_ci':list(ci50),
                      '90pct':float(cover90.mean()),'90pct_group_bootstrap_95_ci':list(ci90)},
    'grid_reference_90pct_coverage':float(ref_cover90.mean()),
    'grid_reference_90pct_group_bootstrap_95_ci':list(ci_ref90),
    'flow_vs_grid_reference': {'median_absolute_difference_in_posterior_median_solar':flow_median_abs_error,
                               'ratio_of_median_90pct_interval_widths_flow_over_grid':median_width_ratio},
    'interpretation': [
        'All coverage and reference comparisons use synthetic held-out draws from the saved Day 7 simulator.',
        'The reference uses the same zero-temperature Nauenberg mass-radius relation, uniform mass prior, and Gaussian radius error as the simulator; it is not an independent physical validation.',
        'This single-parameter flow demonstrates a conditional normalizing-flow approximation for the white-dwarf mass-radius family; it does not prove real-star calibration or correct temperature/composition modeling.'
    ],
    'outputs': ['day15_white_dwarf_conditional_flow.pt','day15_white_dwarf_flow_coverage.csv',
                'day15_white_dwarf_flow_posterior_comparison.csv','day15_white_dwarf_flow_training_history.csv',
                'day15_white_dwarf_flow_summary.json','day15_white_dwarf_flow_diagnostics.png']
}
(OUT/'day15_white_dwarf_flow_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot([h['epoch'] for h in history], [h['train_nll'] for h in history], label='train')
axes[0].plot([h['epoch'] for h in history], [h['validation_nll'] for h in history], label='validation')
axes[0].axvline(best_epoch, color='black', linestyle='--', linewidth=1, label=f'best epoch {best_epoch}')
axes[0].set(xlabel='Epoch', ylabel='Negative log likelihood', title='Conditional flow training')
axes[0].legend()
plot_rng = np.random.default_rng(88)
show = plot_rng.choice(len(truth), size=min(1500, len(truth)), replace=False)
axes[1].scatter(ref_median[show], flow_median[show], s=8, alpha=0.35)
axes[1].plot([MASS_LOW,MASS_HIGH], [MASS_LOW,MASS_HIGH], color='crimson', linewidth=1)
axes[1].set(xlabel='Numerical-grid posterior median (solar masses)', ylabel='Flow posterior median (solar masses)',
            title='Untouched test: flow vs grid reference')
fig.tight_layout()
fig.savefig(OUT/'day15_white_dwarf_flow_diagnostics.png', dpi=160)
plt.show()

print('Saved outputs under:', OUT)
display(coverage)
print('Median absolute flow/grid posterior-median difference (solar masses):', flow_median_abs_error)
print('Median 90% interval-width ratio, flow/grid:', median_width_ratio)
print(json.dumps(summary['interpretation'], indent=2))
